# 5. Modelos del lenguaje (estadísticos)

![](https://lena-voita.github.io/resources/lectures/lang_models/examples/morphosyntax-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

## Objetivo

- Implementar modelos del lenguaje estadísticos
- Aplicaciones

## Modelos del lenguaje

> "Un modelo del lenguaje es un modelo estadístico que asigna probabilidades a cadenas dentro de un lenguaje" - Parafraseando a mi compadre [Jurafsky, 2026](https://web.stanford.edu/~jurafsky/slp3/3.pdf)

$$ \mu = (\Sigma, A, \Pi)$$

Donde:
- $\mu$ es el modelo del lenguaje
- $\Sigma$ es el vocabulario
- $A$ es el tensor que guarda las probabilidades
- $\Pi$ guarda las probabilidades iniciales

- Este modelo busca estimar la probabilidad de una secuencia de tokens
- Pueden ser palabras, caracteres o tokens
- Se pueden considerar varios escenarios para la creación de estos modelos
    - Si podemos estimar la probabilidad de una unidad lingüística (palabras, tokens, oraciones, etc), podemos usarlar de formas insospechadas

### Probabilidad de una oración

- El objetivo es estimar las probabilidades de unidades lingüísticas que reflejen el comportamiento del lenguaje natural
- Esto es, por ejemplo, las oraciones que tengan mayor probabilidad de ocurrir

####  Muchas oraciones probablemente no ocurran en nuestro corpus. ¿Cómo lidiamos con eso❓

#### I saw a cat in a mat

<img src="https://lena-voita.github.io/resources/lectures/lang_models/general/i_saw_a_cat_prob.gif">

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

Sean $y_1, y_2, \dots, y_n$ tokens y $P(y_1, y_2, \dots, y_n)$ la probabilidad de verlos en ese orden. Si aplicamos la regla de la cadena obtenemos:

$$
P(y_1, y_2, \dots, y_n)=P(y_1)\cdot P(y_2|y_1) \cdot P(y_3|y_1, y_2)\cdot\dots\cdot P(y_n|y_1, \dots, y_{n-1})=
        \prod \limits_{t=1}^n P(y_t|y_{<t}).
$$

Con esto modelamos la probabilidad de que un conjunto de tokens ocurran como una **probabilidad condicional** $P(y_n|y_1, \dots, y_{n-1})$. Podemos estimar esta probabilidad de multiples formas:

- N-gramas
- Modelos neuronales

### Sobre bigramas y N-gramas

- Para bigramas tenemos la propiedad de Markov
- Para $n > 2$ las palabras dependen de mas elementos
    - Trigramas
    - 4-gramas
- En general para un modelo de n-gramas se toman en cuenta $n-1$ elementos

![](https://lena-voita.github.io/resources/lectures/lang_models/ngram/example_cut_3gram-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

## Programando nuestros modelos del lenguaje

Utilizaremos un [corpus](https://www.nltk.org/book/ch02.html) en inglés disponible en NLTK

In [1]:
import nltk

nltk.download("gutenberg")
nltk.download("abc")
nltk.download("genesis")
nltk.download("inaugural")
nltk.download("state_union")
nltk.download("webtext")
nltk.download("punkt_tab")

[nltk_data] Downloading package gutenberg to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package abc to /home/umoqnier/nltk_data...
[nltk_data]   Package abc is already up-to-date!
[nltk_data] Downloading package genesis to /home/umoqnier/nltk_data...
[nltk_data]   Package genesis is already up-to-date!
[nltk_data] Downloading package inaugural to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!
[nltk_data] Downloading package state_union to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package state_union is already up-to-date!
[nltk_data] Downloading package webtext to /home/umoqnier/nltk_data...
[nltk_data]   Package webtext is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/umoqnier/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
import numpy as np

In [3]:
from nltk.corpus import abc, genesis, gutenberg, inaugural, state_union, webtext

# Exploración del corpus
total_sents = 0
corpora = []

plaintext_corpora = {
    "abc": abc,
    "Gutenberg": gutenberg,
    "Genesis": genesis,
    "Inaugural": inaugural,
    "State Union": state_union,
    "Web": webtext,
}

for title, corpus in plaintext_corpora.items():
    corpus_sents = corpus.sents()
    corpus_len = len(corpus_sents)
    total_sents += corpus_len
    print(f"{title} sents={corpus_len}")
    corpora.append(corpus_sents)
print(f"Total={total_sents}")

abc sents=29059
Gutenberg sents=98552
Genesis sents=13640
Inaugural sents=5395
State Union sents=17930
Web sents=25733
Total=190309


In [4]:
def preprocess_sent(sent: list[str]) -> list[str]:
    """Función de preprocesamiento

    Agrega tokens de inicio y fin a la oración y normaliza todo a minusculas

    Params
    ------
    sent: list[str]
        Lista de palabras que componen la oración

    Return
    ------
    Las oración preprocesada
    """
    result = [word.lower() for word in sent]
    # Al final de la oración
    result.append("</s>")
    # Al inicio de la oración
    result.insert(0, "<s>")
    return result

In [5]:
print(preprocess_sent(corpora[0][0]))

['<s>', 'pm', 'denies', 'knowledge', 'of', 'awb', 'kickbacks', 'the', 'prime', 'minister', 'has', 'denied', 'he', 'knew', 'awb', 'was', 'paying', 'kickbacks', 'to', 'iraq', 'despite', 'writing', 'to', 'the', 'wheat', 'exporter', 'asking', 'to', 'be', 'kept', 'fully', 'informed', 'on', 'iraq', 'wheat', 'sales', '.', '</s>']


In [6]:
for i, sent in enumerate(corpora[0][:10]):
    print(i, " ".join(sent))

0 PM denies knowledge of AWB kickbacks The Prime Minister has denied he knew AWB was paying kickbacks to Iraq despite writing to the wheat exporter asking to be kept fully informed on Iraq wheat sales .
1 Letters from John Howard and Deputy Prime Minister Mark Vaile to AWB have been released by the Cole inquiry into the oil for food program .
2 In one of the letters Mr Howard asks AWB managing director Andrew Lindberg to remain in close contact with the Government on Iraq wheat sales .
3 The Opposition ' s Gavan O ' Connor says the letter was sent in 2002 , the same time AWB was paying kickbacks to Iraq though a Jordanian trucking company .
4 He says the Government can longer wipe its hands of the illicit payments , which totalled $ 290 million .
5 " The responsibility for this must lay may squarely at the feet of Coalition ministers in trade , agriculture and the Prime Minister ," he said .
6 But the Prime Minister says letters show he was inquiring about the future of wheat sales in 

In [7]:
for i, sent in enumerate(corpora[0][:10]):
    print(i, " ".join(preprocess_sent(sent)))

0 <s> pm denies knowledge of awb kickbacks the prime minister has denied he knew awb was paying kickbacks to iraq despite writing to the wheat exporter asking to be kept fully informed on iraq wheat sales . </s>
1 <s> letters from john howard and deputy prime minister mark vaile to awb have been released by the cole inquiry into the oil for food program . </s>
2 <s> in one of the letters mr howard asks awb managing director andrew lindberg to remain in close contact with the government on iraq wheat sales . </s>
3 <s> the opposition ' s gavan o ' connor says the letter was sent in 2002 , the same time awb was paying kickbacks to iraq though a jordanian trucking company . </s>
4 <s> he says the government can longer wipe its hands of the illicit payments , which totalled $ 290 million . </s>
5 <s> " the responsibility for this must lay may squarely at the feet of coalition ministers in trade , agriculture and the prime minister ," he said . </s>
6 <s> but the prime minister says letters

In [10]:
from nltk import ngrams

for ngram in list(ngrams(preprocess_sent(corpora[0][10]), 2))[:10]:
    print(ngram)

('<s>', 'mr')
('mr', 'geary')
('geary', 'said')
('said', 'he')
('he', 'had')
('had', 'forwarded')
('forwarded', 'the')
('the', 'email')
('email', 'to')
('to', 'two')


### Construyendo el modelo del lenguaje

![](https://imgs.xkcd.com/comics/predictive_models_2x.png)

In [11]:
from collections import Counter, defaultdict  # ❤️‍🔥❤️‍🔥❤️‍🔥

test = defaultdict(Counter)

In [17]:
test[("el", "gato")]["salta"] += 1

In [18]:
test

defaultdict(collections.Counter,
            {('hola', 'que'): Counter({'hace': 1}),
             ('el', 'gato'): Counter({'salta': 2, 'come': 1})})

In [19]:
type NgramModel = defaultdict[tuple[str, str], Counter[str]]

In [20]:
def build_trigram_model(data: list[list[list[str]]]) -> NgramModel:
    # Initialize model as a nested dict with default behavior
    model: NgramModel = defaultdict(Counter)
    for corpus in data:
        for sentence in corpus:
            for w1, w2, w3 in ngrams(preprocess_sent(sentence), 3):
                model[(w1, w2)][w3] += 1
    return model

In [21]:
%%time
trigram_model = build_trigram_model(corpora)

CPU times: user 11.8 s, sys: 440 ms, total: 12.3 s
Wall time: 12.3 s


In [23]:
trigram_model["<s>", "in"]

Counter({'the': 603,
         'a': 149,
         'this': 118,
         'fact': 53,
         'addition': 47,
         'that': 45,
         'one': 38,
         'our': 38,
         'all': 38,
         'my': 33,
         'short': 27,
         'such': 25,
         'other': 21,
         'these': 21,
         'some': 20,
         'an': 20,
         'general': 18,
         'another': 17,
         'his': 17,
         'their': 13,
         'spite': 12,
         'those': 12,
         'vain': 12,
         'order': 11,
         'recent': 11,
         'every': 11,
         'no': 10,
         'less': 9,
         'its': 8,
         'particular': 8,
         'each': 8,
         'what': 8,
         'your': 8,
         'new': 7,
         'front': 7,
         'just': 6,
         'australia': 6,
         'victoria': 6,
         'many': 6,
         'most': 6,
         'her': 6,
         'two': 6,
         'any': 6,
         'good': 6,
         'view': 6,
         'america': 6,
         'south': 5,
         

### Calculo de probabilidades con conteos de trigramas

In [24]:
for i, key in enumerate(trigram_model):
    print(key)
    if i == 5:
        break

('<s>', 'pm')
('pm', 'denies')
('denies', 'knowledge')
('knowledge', 'of')
('of', 'awb')
('awb', 'kickbacks')


In [25]:
def calculate_model_probs(model: NgramModel) -> NgramModel:
    model_probs: NgramModel = defaultdict(Counter)
    # Por cada prefijo del modelo
    for prefix in model:
        # Todas las veces que ocurre prefix seguido de cualquier palabra
        total = float(sum(model[prefix].values()))
        # Por cada palabra w que haya ocurrido con prefix
        for w in model[prefix]:
            # Obtenemos la probabilidad
            model_probs[prefix][w] = model[prefix][w] / total
    return model_probs

## Smoothing 🥤

![](https://lena-voita.github.io/resources/lectures/lang_models/ngram/prob_cat_on_a_mat-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

¿Qué pasaría se no sucede la secuencia de tokens del númerador/denominador? Para evitar este problema se utiliza una técnica llamada *smoothing* que redistribuye la función de masa de probabilidad.

#### ¿Cómo agregamos smoothing❓

La forma más sencilla es pretender que vimos al menos una vez todos los n-gramas. Esto es sumar 1 a todas las cuentas. Algo más general sería agregar una cantidad $\delta$:

$$
P(mat| cat\ on\ a) = \frac{\delta + N(cat\ on\ a\ mat)}{\delta \cdot |V| + N(cat\ on\ a)}
$$

#### Ejercicio 🤹🏽: Implementa el smoothing de Laplace agregando $\delta$ a todas las cuentas de los n-gramas

- *Hint*: Obten todos los tokens, despues el vocabulario y el tamaño del vocabulario

In [26]:
# Calcula el vocabulario (tipos) acá
TOKENS = []
for _, corpus in plaintext_corpora.items():
    for sent in corpus.sents():
        for word in sent:
            TOKENS.append(word.lower())
VOCABULARY = set(TOKENS)
# +2 por los tokens <s> y </s>
VOCABULARY_SIZE = len(VOCABULARY) + 2

In [28]:
VOCABULARY_SIZE

85406

In [31]:
len(trigram_model)

1045956

In [29]:
def calculate_smooth_probs(model: NgramModel, vocab_size: int, delta: float = 1.0) -> NgramModel:
    model_probs = defaultdict(Counter)
    for prefix in model:
        total = float(sum(model[prefix].values()))
        for w in model[prefix]:
            model_probs[prefix][w] = (model[prefix][w] + delta) / (
                total + delta * vocab_size
            )
    return model_probs


In [32]:
trigram_probs = calculate_model_probs(trigram_model)

In [33]:
trigram_smooth = calculate_smooth_probs(trigram_model, VOCABULARY_SIZE)

In [34]:
sorted(dict(trigram_probs["<s>", "the"]).items(), key=lambda x: -1 * x[1])

[('researchers', 0.02091516103692066),
 ('first', 0.012666928515318147),
 ('little', 0.009819324430479183),
 ('study', 0.009524744697564808),
 ('new', 0.008837391987431265),
 ('federal', 0.007953652788688138),
 ('australian', 0.007953652788688138),
 ('man', 0.007462686567164179),
 ('scientists', 0.007364493322859387),
 ('next', 0.006677140612725845),
 ('government', 0.006382560879811469),
 ('other', 0.005989787902592302),
 ('two', 0.005793401413982718),
 ('national', 0.0054988216810683424),
 ('company', 0.0054988216810683424),
 ('only', 0.005400628436763551),
 ('team', 0.0053024351924587584),
 ('old', 0.0053024351924587584),
 ('american', 0.0053024351924587584),
 ('world', 0.004615082482325216),
 ('people', 0.004516889238020424),
 ('research', 0.004418695993715633),
 ('great', 0.004418695993715633),
 ('second', 0.00432050274941084),
 ('sons', 0.004222309505106049),
 ('united', 0.004124116260801257),
 ('report', 0.004025923016496465),
 ('findings', 0.004025923016496465),
 ('time', 0.003

In [35]:
sorted(dict(trigram_smooth["<s>", "the"]).items(), key=lambda x: -1 * x[1])

[('researchers', 0.0022387279004079923),
 ('first', 0.0013599748927712104),
 ('little', 0.0010565958782299404),
 ('study', 0.0010252118422429124),
 ('new', 0.0009519824249398473),
 ('federal', 0.0008578303169787635),
 ('australian', 0.0008578303169787635),
 ('man', 0.000805523590333717),
 ('scientists', 0.0007950622450047076),
 ('next', 0.0007218328277016424),
 ('government', 0.0006904487917146145),
 ('other', 0.0006486034103985773),
 ('two', 0.0006276807197405586),
 ('national', 0.0005962966837535307),
 ('company', 0.0005962966837535307),
 ('only', 0.0005858353384245214),
 ('team', 0.0005753739930955121),
 ('old', 0.0005753739930955121),
 ('american', 0.0005753739930955121),
 ('world', 0.0005021445757924469),
 ('people', 0.0004916832304634376),
 ('research', 0.0004812218851344283),
 ('great', 0.0004812218851344283),
 ('second', 0.000470760539805419),
 ('sons', 0.00046029919447640965),
 ('united', 0.00044983784914740037),
 ('report', 0.00043937650381839103),
 ('findings', 0.00043937650

In [38]:
trigram_smooth["<s>", "the"]

Counter({'researchers': 0.0022387279004079923,
         'first': 0.0013599748927712104,
         'little': 0.0010565958782299404,
         'study': 0.0010252118422429124,
         'new': 0.0009519824249398473,
         'federal': 0.0008578303169787635,
         'australian': 0.0008578303169787635,
         'man': 0.000805523590333717,
         'scientists': 0.0007950622450047076,
         'next': 0.0007218328277016424,
         'government': 0.0006904487917146145,
         'other': 0.0006486034103985773,
         'two': 0.0006276807197405586,
         'national': 0.0005962966837535307,
         'company': 0.0005962966837535307,
         'only': 0.0005858353384245214,
         'team': 0.0005753739930955121,
         'old': 0.0005753739930955121,
         'american': 0.0005753739930955121,
         'world': 0.0005021445757924469,
         'people': 0.0004916832304634376,
         'research': 0.0004812218851344283,
         'great': 0.0004812218851344283,
         'second': 0.000470760539

### Aplicaciones

- Generación de texto
- Completado de texto
- Speech To Text (STT)

![](https://lena-voita.github.io/resources/lectures/lang_models/examples/suggest-min.png)

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

### Generación de texto 📨

<video src="https://lena-voita.github.io/resources/lectures/lang_models/general/generation_example.mp4" controls loop>

> Tomado de [Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html)

In [ ]:
def get_likely_words(
    model: NgramModel, context: str, top_count: int = 10
) -> list[tuple]:
    """Dado un contexto obtiene las palabras más probables

    Params
    ------
    model: NgramModel
        Modelo con sus probabilidades calculadas
    context: str
        Contexto con el cual calcular las palabras más probables siguientes
    top_count: int
        Cantidad de palabras más probables. Default 10
    """
    history = tuple(context.split())
    return model[history].most_common(top_count)

In [41]:
get_likely_words(trigram_probs, "<s> the", top_count=10)

[('researchers', 0.02091516103692066),
 ('first', 0.012666928515318147),
 ('little', 0.009819324430479183),
 ('study', 0.009524744697564808),
 ('new', 0.008837391987431265),
 ('federal', 0.007953652788688138),
 ('australian', 0.007953652788688138),
 ('man', 0.007462686567164179),
 ('scientists', 0.007364493322859387),
 ('next', 0.006677140612725845)]

In [66]:
import random

# Strategy here
def get_next_word(words: list[tuple]) -> str:
    return words[0][0]

def get_next_random_word(words: list) -> str:
    if not words:
        return "</s>"
    return random.choice(words)[0]

def get_next_top_p_word(words: list[tuple[str, float]], p: float = 0.8) -> str:
    """
    Selecciona la siguiente palabra utilizando Nucleus Sampling (Top-p).
    
    Params:
    - words: Lista de tuplas (palabra, probabilidad).
    - p: Umbral de masa de probabilidad acumulada (típicamente entre 0.8 y 0.95).
    """
    if not words:
        return "</s>"
        
    # Aseguramos que la lista esté ordenada de mayor a menor probabilidad
    # sorted_words = sorted(words, key=lambda x: x[1], reverse=True)
    
    valid_words = []
    valid_probs = []
    cumulative_prob = 0.0
    
    # Recolectamos palabras hasta que la suma de probabilidades alcance el umbral 'p'
    for word, prob in words:
        valid_words.append(word)
        valid_probs.append(prob)
        cumulative_prob += prob
        
        if cumulative_prob >= p:
            break
            
    # Muestreamos una palabra del subconjunto válido (núcleo) usando sus probabilidades como pesos.
    # random.choices devuelve una lista, por lo que extraemos el elemento [0]
    return random.choices(valid_words, weights=valid_probs, k=1)[0]


In [46]:
get_next_word(get_likely_words(trigram_probs, "emma was", 50))

'obliged'

In [50]:
sentence = "<s> the"
for i in range(10):
    print(i, get_next_random_word(get_likely_words(trigram_probs, sentence, 50)))

0 state
1 whole
2 only
3 best
4 second
5 best
6 findings
7 men
8 national
9 first


#### Ejercicio 🤺: Genera lenguaje


- Utilizando el modelo de trigramas, diseña una estrategia para generación del lenguaje
- Implementa la fución `generate_text()` que reciba un modelo de n-gramas y una historia y genere texto utilizando la estrategia implementada.
- Agrega los parámetros que consideres necesarios a tu función de generación de texto

**Ejemplo:**

```python
sentence = "god was"
generate_text(trigram_probs, sentence)
>> god was evil 🐲
```

In [51]:
import time
from random import uniform
from typing import Callable

def generate_text(
    model: NgramModel,
    history: str,
    strategy: Callable,
    tokens_count: int = 0,
    top_n: int = 10,
    max_tokens: int = 50,
    use_gpu: bool = False,
):
    next_word = strategy(get_likely_words(model, history, top_count=top_n))

    if not use_gpu:
        time.sleep(uniform(0.1, 0.3))
    
    print(next_word, end=" ")
    
    tokens_count += 1
    
    if tokens_count == max_tokens or next_word == "</s>":
        return

    new_history = history.split()[-1] + " " + next_word
    
    return generate_text(
        model,
        new_history,
        strategy,
        tokens_count,
        top_n,
        max_tokens,
        use_gpu,
    )


In [69]:
sentence = "science is"
print(sentence, end=" ")
generate_text(trigram_probs, sentence, get_next_top_p_word, 0, max_tokens=10, top_n=15, use_gpu=False)

science is very slow ; but the lord thy god shall bless 

### Calculando la probabilidad de una oración

In [70]:
def calculate_sentence_prob(sentence: list[str], model: NgramModel) -> float:
    prob = 0
    for w1, w2, w3 in ngrams(sentence, n=3):
        try:
            prob += np.log(model[w1, w2][w3])
        except KeyError:
            # OOV
            prob += 0.0
    return prob

In [71]:
nltk.download("reuters")

[nltk_data] Downloading package reuters to /home/umoqnier/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


True

In [72]:
from nltk.corpus import reuters

In [73]:
news_sentence = preprocess_sent(reuters.sents()[6701])
gutenberg_sentence = preprocess_sent(gutenberg.sents()[100])
sentences = [news_sentence, gutenberg_sentence, preprocess_sent(gutenberg.sents()[101])]

for sent in sentences:
    print(f"PROB={calculate_sentence_prob(sent, trigram_smooth)}: '{' '.join(sent)}'")

PROB=-75.97054951953956: '<s> " the future belongs to the brave . </s>'
PROB=-171.24531593491298: '<s> you do not think i could mean _you_ , or suppose mr . knightley to mean _you_ . </s>'
PROB=-47.58946760523992: '<s> what a horrible idea ! </s>'


In [ ]:
i = 0

for j, sent in enumerate(reuters.sents()):
    sent = preprocess_sent(sent)
    if calculate_sentence_prob(sent, trigram_smooth) != -np.inf:
        print(
            f"{j} PROB={calculate_sentence_prob(sent, trigram_smooth)}: '{' '.join(sent)}'"
        )
        i += 1
    if i > 30:
        break

### Usando un dataset grandesito

In [74]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus-100", "en-es", split="train")

In [77]:
from nltk import word_tokenize


def build_vocabulary(dataset, vocab_size=30000):
    """
    Construye los diccionarios de mapeo para limitar el consumo de memoria.
    """
    word_counts = Counter()
    
    # Contamos frecuencias (solo procesamos el texto en español)
    for item in dataset:
        # Tokenización básica por espacios (puedes usar nltk.word_tokenize si lo prefieres)
        tokens = word_tokenize(item['translation']['es'].lower(), language="spanish")
        word_counts.update(tokens)
        
    # Nos quedamos solo con las 'vocab_size' palabras más frecuentes
    most_common = word_counts.most_common(vocab_size)
    
    # Inicializamos con nuestros tokens especiales
    word2idx = {"<UNK>": 0, "<s>": 1, "</s>": 2}
    idx2word = {0: "<UNK>", 1: "<s>", 2: "</s>"}
    
    # Asignamos un ID único a cada palabra del vocabulario
    for idx, (word, _) in enumerate(most_common, start=3):
        word2idx[word] = idx
        idx2word[idx] = word
        
    return word2idx, idx2word


In [ ]:
# Construimos el vocabulario (Esto puede tomar un minuto)
word2idx, idx2word = build_vocabulary(dataset, vocab_size=1_000_000)

In [80]:
VOCAB_INT_SIZE = len(word2idx)
print(f"Vocabulario creado. Tamaño: {len(word2idx)}")


Vocabulario creado. Tamaño: 291448


In [81]:
def encode_sentence(tokens: list[str], word2idx: dict) -> list[int]:
    """Convierte una lista de strings a enteros usando el diccionario."""
    return [word2idx.get(word, word2idx["<UNK>"]) for word in tokens]

In [82]:
def build_integer_trigram_model(dataset, word2idx) -> NgramModel:
    """
    Construye el modelo de trigramas almacenando únicamente enteros.
    """
    # La estructura sigue siendo la misma, pero ahora almacena int -> int
    model: NgramModel = defaultdict(Counter)
    
    for item in dataset:
        raw_tokens = word_tokenize(item['translation']['es'].lower(), language="spanish")
        
        # Agregamos los IDs de los tokens <s> y </s>
        encoded_tokens = [word2idx["<s>"]] + encode_sentence(raw_tokens, word2idx) + [word2idx["</s>"]]
        
        # Construimos el modelo con las secuencias numéricas
        for w1, w2, w3 in ngrams(encoded_tokens, 3):
            model[(w1, w2)][w3] += 1
            
    return model


In [83]:
# Entrenamos el modelo "eficiente" en memoria (esto tomará ~2 minutos)
integer_trigram_model = build_integer_trigram_model(dataset, word2idx)

In [84]:
int_prob_trigram_model = calculate_model_probs(integer_trigram_model)

In [85]:
def generate_text_int(
    model: dict,
    context_idx: tuple[int, int],  # El historial ahora es una tupla de IDs numéricos
    idx2word: dict,                # Diccionario para decodificar
    strategy: Callable,
    tokens_count: int = 0,
    top_n: int = 10,
    max_tokens: int = 50,
    use_gpu: bool = False,
):
    predictions = model.get(context_idx, {})
    
    # Si el modelo llega a un callejón sin salida, nos detenemos
    if not predictions:
        return

    # Obtenemos los candidatos principales y normalizamos sus conteos a probabilidades
    top_candidates = predictions.most_common(top_n)
    
    # La estrategia elige el siguiente ID numérico
    next_word_idx = strategy(top_candidates)
    next_word_str = idx2word[next_word_idx]
    
    if not use_gpu:
        time.sleep(random.uniform(0.1, 0.3))
        
    print(next_word_str, end=" ")
    tokens_count += 1
    
    if tokens_count == max_tokens or next_word_str == "</s>":
        return
        
    # Actualizamos la ventana (desplazamos a la izquierda y añadimos el nuevo ID)
    new_context_idx = (context_idx[1], next_word_idx)
    
    return generate_text_int(
        model,
        new_context_idx,
        idx2word,
        strategy,
        tokens_count,
        top_n,
        max_tokens,
        use_gpu,
    )

In [89]:
text_prompt = "el gobierno"
# Procesamos la entrada del usuario
tokens = text_prompt.lower().split()
print(text_prompt, end=" ")
    
# Extraemos las dos últimas palabras del prompt y las codificamos
idxs = encode_sentence(tokens, word2idx)

w1 = idxs[-2]
w2 = idxs[-1]
context_idx = (w1, w2)

generate_text_int(
    model=int_prob_trigram_model,
    context_idx=context_idx,
    idx2word=idx2word,
    strategy=get_next_random_word,
    max_tokens=20
)

el gobierno de los programas para el mantenimiento a la luz de su vida , que los países bajos de azúcar ? 


## Referencias

- [Maravilloso curso de Lena Voita](https://lena-voita.github.io/nlp_course/language_modeling.html) ⚡